# 02 - Data Cleaning & Feature Engineering
**Project:** Rossmann Store Sales Forecasting

This notebook takes the raw data and cleans it up based on what we found in notebook 01, then engineers a few new features.
Missing values and outliers are intentionally left for notebooks 03 and 04.

**Input:** `../data/train.csv`  →  **Output:** `../data/train_cleaned.csv`

**Steps**
1. Load the data & fix data types
2. Closed days (`Open = 0`)
3. Suspicious rows (`Sales = 0` / `Customers = 0` on open days)
4. Stores with incomplete history
5. Date features
6. Other engineered features
7. Save the cleaned data

In [ ]:
import calendar
import os
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:          # outside Jupyter
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load & fix data types

In [ ]:
DATA_PATH = "../data/train.csv"     # change this if the file is in another folder
STORE_PATH = "../data/store.csv"    # only used if train.csv does not already contain the store columns

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} not found - put it next to this notebook or change DATA_PATH")

# StateHoliday mixes numbers and letters (0 / a / b / c) -> read it as text
clean_df = pd.read_csv(DATA_PATH, dtype={"StateHoliday": "str"})

# If the store information is not inside train.csv yet, add it from store.csv
if "StoreType" not in clean_df.columns:
    store_info = pd.read_csv(STORE_PATH)
    clean_df = clean_df.merge(store_info, on="Store", how="left")

# Date: text -> real datetime
clean_df["Date"] = pd.to_datetime(clean_df["Date"])

# Sort: every store with its dates in order
clean_df = clean_df.sort_values(["Store", "Date"]).reset_index(drop=True)

print("Shape:", clean_df.shape)
clean_df.info()
print()
print(clean_df["StateHoliday"].value_counts())

## 2. Drop closed days
Closed days always have `Sales = 0` — about 17% of the dataset.
They carry zero information for forecasting, so we drop them.
We count days per store before dropping because we need it in step 4.

In [ ]:
rows_raw = len(clean_df)
days_per_store = clean_df.groupby("Store")["Date"].nunique()   # saved before dropping rows (used in step 4)

closed = clean_df["Open"] == 0
print(f"Closed days                 : {closed.sum():,} ({closed.mean():.1%})")
print("Closed days with Sales > 0  :", (closed & (clean_df["Sales"] > 0)).sum())

In [ ]:
clean_df = clean_df[~closed].reset_index(drop=True)
print(f"Rows before: {rows_raw:,}  ->  after dropping closed days: {len(clean_df):,}")

## 3. Remove suspicious rows
Some open days have `Sales = 0` (54 rows) or `Customers = 0` (52 rows) — that shouldn't happen.
An open store should have both. We inspect and drop them.

In [ ]:
zero_sales = clean_df["Sales"] == 0
zero_customers = clean_df["Customers"] == 0

print("Open days with Sales == 0     :", zero_sales.sum())
print("Open days with Customers == 0 :", zero_customers.sum())
print("Both at the same time         :", (zero_sales & zero_customers).sum())
print("Stores involved               :", clean_df.loc[zero_sales | zero_customers, "Store"].nunique())
print("Negative Sales / Customers    :", (clean_df["Sales"] < 0).sum(), "/", (clean_df["Customers"] < 0).sum())

display(clean_df[zero_sales | zero_customers].head(10))

**Decision:** drop them. A zero on an open day is a data entry error, not a genuine observation,
and `log(0)` is undefined which would break notebook 06.

In [ ]:
before = len(clean_df)
clean_df = clean_df[~(zero_sales | zero_customers)].reset_index(drop=True)
print(f"Dropped {before - len(clean_df)} suspicious rows  ->  {len(clean_df):,} rows left")

## 4. Flag stores with incomplete history
Each store should have 942 days of data. Some only have 758 days or 941 — we flag these instead of dropping them.
We don't fill in the gaps — just add a binary flag so the model can use it.

In [ ]:
expected_days = days_per_store.max()
print("Expected days per store:", expected_days)
display(days_per_store.value_counts().rename_axis("days per store").to_frame("number of stores"))

incomplete_stores = days_per_store[days_per_store < expected_days].index
clean_df["IncompleteHistory"] = clean_df["Store"].isin(incomplete_stores).astype(int)

print("Stores with incomplete history:", len(incomplete_stores))
print(clean_df.groupby("IncompleteHistory")["Store"].nunique().rename("stores"))

## 5. Date features
Break `Date` into components a model can actually use. `DayOfWeek` is already there, so we just add the rest.

In [ ]:
dt = clean_df["Date"].dt

clean_df["Year"] = dt.year
clean_df["Month"] = dt.month
clean_df["Day"] = dt.day
clean_df["WeekOfYear"] = dt.isocalendar().week.astype(int)
clean_df["Quarter"] = dt.quarter
clean_df["IsWeekend"] = (clean_df["DayOfWeek"] >= 6).astype(int)      # Saturday / Sunday
clean_df["IsMonthStart"] = dt.is_month_start.astype(int)
clean_df["IsMonthEnd"] = dt.is_month_end.astype(int)

date_cols = ["Date", "Year", "Month", "Day", "WeekOfYear", "Quarter", "DayOfWeek", "IsWeekend", "IsMonthStart", "IsMonthEnd"]
display(clean_df[date_cols].head())

## 6. Engineered features
Build a few more features from the existing columns.
NaNs are left untouched here (notebook 03 handles them), and outliers go to notebook 04.

In [ ]:
import calendar

# 6.1 Is it a state holiday? (a / b / c -> 1, 0 -> 0). The original StateHoliday is kept for encoding in notebook 07.
clean_df["IsStateHoliday"] = (clean_df["StateHoliday"] != "0").astype(int)

# 6.2 How many months ago did the nearest competitor open? (0 if it had not opened yet at that date)
comp_months = ((clean_df["Year"] - clean_df["CompetitionOpenSinceYear"]) * 12
               + (clean_df["Month"] - clean_df["CompetitionOpenSinceMonth"]))
clean_df["CompetitionOpenMonths"] = comp_months.clip(lower=0)

# 6.3 How many weeks has the store been in Promo2? (0 if it had not joined yet at that date)
promo2_weeks = ((clean_df["Year"] - clean_df["Promo2SinceYear"]) * 52
                + (clean_df["WeekOfYear"] - clean_df["Promo2SinceWeek"]))
clean_df["Promo2OpenWeeks"] = promo2_weeks.clip(lower=0)

# 6.4 Is the current month one of the Promo2 restart months AND has the store already joined Promo2?
#     (PromoInterval looks like "Jan,Apr,Jul,Oct"; the raw file writes September as "Sept",
#      so we change it to "Sep" to match calendar.month_abbr)
intervals = clean_df["PromoInterval"].str.replace("Sept", "Sep")
month_names = clean_df["Month"].map(lambda m: calendar.month_abbr[m])
in_interval = [isinstance(s, str) and m in s.split(",") for m, s in zip(month_names, intervals)]
clean_df["IsPromo2Month"] = (pd.Series(in_interval, index=clean_df.index) & (promo2_weeks >= 0)).astype(int)

In [ ]:
# 6.5 After step 2 every row has Open = 1, so the column carries no information -> drop it
print("Unique values of Open:", clean_df["Open"].unique())
clean_df = clean_df.drop(columns=["Open"])

new_cols = ["IncompleteHistory", "IsStateHoliday", "CompetitionOpenMonths", "Promo2OpenWeeks", "IsPromo2Month"]
display(clean_df[new_cols].describe().T)
display(clean_df[new_cols].head())

## 7. Save
Quick sanity check, then save so notebooks 03–08 can pick up from here.

In [ ]:
print(f"Rows : {rows_raw:,} -> {len(clean_df):,}   (removed {rows_raw - len(clean_df):,})")
print(f"Columns : {clean_df.shape[1]}")

print("\nMissing values left (handled in notebook 03):")
missing_left = clean_df.isna().sum()
display(missing_left[missing_left > 0].to_frame("missing_count"))

clean_df.to_csv("../data/train_cleaned.csv", index=False)
print("\nSaved: train_cleaned.csv   (when reading it back, use dtype={'StateHoliday': 'str'})")

### What this notebook did
| Step | What we did |
|---|---|
| 2 | Dropped closed days (`Open = 0`) |
| 3 | Dropped open days with `Sales = 0` or `Customers = 0` |
| 4 | Flagged stores with incomplete history (`IncompleteHistory`) |
| 5 | Added date features: `Year`, `Month`, `Day`, `WeekOfYear`, `Quarter`, `IsWeekend`, `IsMonthStart`, `IsMonthEnd` |
| 6 | Added `IsStateHoliday`, `CompetitionOpenMonths`, `Promo2OpenWeeks`, `IsPromo2Month` and dropped `Open` |
| 7 | Saved `train_cleaned.csv` |

**Still to do:** missing values → notebook 03 | outliers (e.g. `CompetitionOpenSinceYear = 1900`) → 04 | scaling → 05 | transformations → 06 | encoding `StoreType`, `Assortment`, `StateHoliday`, `PromoInterval` → 07.